# Swiss Law Perfect-Recall — Generalized v1

**Goal.** Retrieve every law citation in `laws_de.csv` that an expert would mark gold for an arbitrary English Swiss-law query, with no per-query hardcoding and no use of `train.csv` (German queries — wrong language and wrong distribution per `personal_observations.md` Obs 4).

**Why this pipeline can hit law-only recall ≈ 1.0 on val while dense embedding cannot (Obs 3).** Every val law-gold citation belongs to exactly one of four bridge types (study: `research/val_law_gold_anatomy_2026-05-23.md`):

| Bridge | % of val law gold | Retrieval channel |
|---|---:|---|
| A — explicit `Art. N CODE` in query | ~4% | **Channel 1** — regex + DE/FR/IT code-alias table |
| B — English concept ↔ German doctrine `title` heading | ~54% | **Channel 2** — title-column BM25 driven by LLM-generated German query + doctrine vocab |
| C — universal procedural apparatus for the legal area | ~28% | **Channel 3** — LLM area classification + procedural-heading filter on corpus |
| D — one-hop references from substantive articles | ~15% | **Channel 4** — regex `Art. N CODE` on candidate text bodies |

Channels are unioned, reranked by Qwen3-Reranker-8B for precision, then selected by a multi-channel rule.

**Inputs required on Drive (all already present from your v12 pipeline):**

| Path | Purpose |
|---|---|
| `Omnilex-Agentic-Retrieval-Competition/data/laws_de.csv` | corpus (175,933 rows; columns `citation`, `text`, `title`) |
| `Omnilex-Agentic-Retrieval-Competition/data/val.csv` | val queries + gold (for diagnostic evaluation only — not used as signal) |
| `Omnilex-Agentic-Retrieval-Competition/data/test.csv` | test queries (gold hidden; pipeline runs identically on these) |

**No new uploads required.** Qwen3-Reranker-8B and Qwen3-8B are downloaded by `transformers` on first run.

**Outputs cached to Drive on first run** (subsequent runs reuse them, ~1 min instead of ~30):

- `retrieval/perfect_law_recall_v1/title_bm25.pkl` — BM25 over the `title` column
- `retrieval/perfect_law_recall_v1/llm_expansions.json` — Qwen3-8B query expansions per qid
- `retrieval/perfect_law_recall_v1/reranker_cache.json` — Qwen3-Reranker scores per (qid, citation)
- `retrieval/perfect_law_recall_v1/submission.csv` — final predictions


---

## Architecture

```
                  English Swiss-law query
                            │
        ┌───────────────────┼────────────────────┐
        ▼                   ▼                    ▼
 Channel 1                Qwen3-8B            Channel 4
 statute regex          query expander       (later — after
+ DE/FR/IT alias        → JSON: {area,        Channel 1 lands)
 → direct lookup          statute_mentions,   regex Art. N CODE
 in laws_de.csv          doctrine_concepts,   in candidate text
                          german_query}        → laws_de lookup
                              │
             ┌────────────────┼────────────────┐
             ▼                ▼                ▼
        Channel 2a       Channel 2b       Channel 3
        Title-BM25       Doctrine        Procedural apparatus:
        with             headings →      area's code family ∩
        german_query     citations       procedural-heading
                                          patterns
                              │
                              ▼
                    Union → candidate pool (≈200-500/query)
                              │
                              ▼
                 Qwen3-Reranker-8B yes/no logit softmax
                              │
                              ▼
                  Multi-channel selection rule:
                    - statute_parser → always include
                    - procedural + reranker > 0.3 → include
                    - title_bm25 + reranker > 0.5 → include
                    - 2+ channels + reranker > 0.4 → include
                    - reranker > 0.7 alone → include
                              │
                              ▼
                       apply_proper_case
                              │
                              ▼
                        submission.csv
```

Note: this notebook predicts **only laws_de citations** (court paragraphs are a separate pipeline). On val this caps whole-gold F1 at 0.59 (court gold is 40.6% of val gold by construction). But law-only precision and recall both target ~1.0.


---

## 1. Setup


In [ ]:
!pip install -q rank_bm25 transformers>=4.51 accelerate tqdm pandas pyarrow numpy

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    IN_COLAB = True
    print('✓ Drive mounted.')
except ImportError:
    IN_COLAB = False
    print('✓ Local mode (not Colab).')

In [ ]:
import os, re, gc, sys, json, math, time, pickle, random, datetime
from pathlib import Path
from collections import defaultdict, Counter
from dataclasses import dataclass, field, fields as dc_fields

import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm

os.environ['PYTHONIOENCODING'] = 'utf-8'
random.seed(42); np.random.seed(42)

# ── Paths ──────────────────────────────────────────────────────────
if IN_COLAB:
    DRIVE_BASE = Path('/content/drive/MyDrive/Omnilex-Agentic-Retrieval-Competition')
else:
    DRIVE_BASE = Path(r'E:\swiss_citation_extraction')  # adjust for local runs

DATA_DIR    = DRIVE_BASE / 'data'
CACHE_DIR   = DRIVE_BASE / 'retrieval' / 'perfect_law_recall_v1'
OUTPUT_DIR  = CACHE_DIR
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ── Models ─────────────────────────────────────────────────────────
EXPANDER_MODEL  = 'Qwen/Qwen3-8B'
RERANKER_MODEL  = 'Qwen/Qwen3-Reranker-8B'
RERANKER_BATCH  = 8
DEVICE          = 'auto'

# ── Channel parameters ─────────────────────────────────────────────
TITLE_BM25_TOP_K        = 200
BODY_BM25_TOP_K         = 100        # safety net using the existing text-BM25 pattern
REF_EXPAND_DEPTH        = 1          # one-hop expansion only
POOL_HARD_CAP           = 600        # absolute upper bound on candidates per query

# ── Selection thresholds (calibrated to each channel's reliability) ─
TH_ALWAYS_INCLUDE       = ['statute_parser']             # exact match → trust unconditionally
TH_PROCEDURAL           = 0.30
TH_TITLE_BM25           = 0.50
TH_REF_EXPANSION        = 0.55
TH_MULTI_CHANNEL        = 0.40       # 2+ channels surfaced → relax
TH_SOLO_RERANKER        = 0.70       # only-reranker → high bar
MULTI_CHANNEL_MIN_HITS  = 2

def log(*args, end='
'): print(*args, end=end, flush=True)
def section(t): log('\n' + '='*70 + (f'\n  {t}\n' + '='*70 if t else ''))

log(f'DRIVE_BASE: {DRIVE_BASE}')
log(f'CACHE_DIR : {CACHE_DIR}')

---

## 2. Public Swiss-law metadata

These tables are sourced from the front matter of the public Federal Codes (`admin.ch`), not from val. They are auditable and would be the same for any consumer of Swiss federal law. Two tables:

1. **`CODE_ALIASES`** — DE / FR / IT / EN aliases for every Swiss federal code that appears in `laws_de.csv`. Used by Channel 1 to canonicalize cross-lingual statute mentions to the DE form the corpus stores.
2. **`PROCEDURAL_HEADING_PATTERNS`** — universal procedural-heading regex patterns. Any `laws_de` row whose `title` heading matches one of these is a candidate procedural-apparatus article. Mined by skimming the BGG / StPO / ZPO / SchKG / ATSG tables of contents.
3. **`LEGAL_AREA_CODES`** — coarse mapping from a legal-area label (assigned by the LLM) to the set of code families that govern it. Used by Channel 3 to scope the procedural-apparatus search to the relevant codes.

None of the three references val gold.

In [ ]:
# ── Cross-lingual code aliases (DE / FR / IT / EN → canonical DE) ──────────
# Each lhs is an abbreviation as it might appear in a query (case-sensitive on
# the canonical Swiss-German form, fuzzy-matched on the others).
CODE_ALIASES = {
    # ── Procedural ─────────────────────────────────────────────────────
    'StPO': 'StPO',  'CPP': 'StPO',                  # Strafprozessordnung / Code de procédure pénale / Codice di procedura penale
    'ZPO': 'ZPO',    'CPC': 'ZPO',                   # Zivilprozessordnung / Code de procédure civile / Codice di procedura civile
    'BGG': 'BGG',    'LTF': 'BGG',                   # Bundesgerichtsgesetz / Loi sur le Tribunal fédéral / Legge sul Tribunale federale
    'SchKG': 'SchKG','LP': 'SchKG',   'LEF': 'SchKG',# Schuldbetreibungs- und Konkursgesetz / Loi sur la poursuite / Legge esecuzione
    'MStP': 'MStP',                                   # Militärstrafprozess
    'JStPO': 'JStPO','PPMin': 'JStPO',               # Jugendstrafprozessordnung
    'VwVG': 'VwVG',  'PA': 'VwVG',                   # Verwaltungsverfahrensgesetz
    # ── Substantive ────────────────────────────────────────────────────
    'ZGB': 'ZGB',    'CC': 'ZGB',                    # Zivilgesetzbuch / Code civil / Codice civile
    'OR': 'OR',      'CO': 'OR',                     # Obligationenrecht / Code des obligations / Codice delle obbligazioni
    'StGB': 'StGB',  'CP': 'StGB',                   # Strafgesetzbuch / Code pénal / Codice penale (also EN "Criminal Code")
    'MStG': 'MStG',                                   # Militärstrafgesetz
    'JStG': 'JStG',  'DPMin': 'JStG',                # Jugendstrafgesetz
    'IPRG': 'IPRG',  'LDIP': 'IPRG',                 # IPR-Gesetz / Loi sur le droit international privé
    # ── Social insurance ───────────────────────────────────────────────
    'ATSG': 'ATSG',  'LPGA': 'ATSG',                 # Allgemeiner Teil des Sozialversicherungsrechts
    'IVG': 'IVG',    'LAI': 'IVG',                   # Invalidenversicherung
    'AHVG': 'AHVG',  'LAVS': 'AHVG',                 # Alters- und Hinterlassenenversicherung
    'UVG': 'UVG',    'LAA': 'UVG',    'LAINF': 'UVG',# Unfallversicherung
    'BVG': 'BVG',    'LPP': 'BVG',                   # Berufliche Vorsorge
    'KVG': 'KVG',    'LAMal': 'KVG',                 # Krankenversicherung
    'EOG': 'EOG',    'LAPG': 'EOG',                  # Erwerbsersatzordnung
    'AVIG': 'AVIG',  'LACI': 'AVIG',                 # Arbeitslosenversicherung
    'FZG': 'FZG',    'LFLP': 'FZG',                  # Freizügigkeitsgesetz
    # ── Constitutional / federal-court organization ────────────────────
    'BV': 'BV',      'Cst': 'BV',     'Cost': 'BV',  # Bundesverfassung / Constitution / Costituzione
    'StBOG': 'StBOG','LOAP': 'StBOG',                # Strafbehördenorganisationsgesetz
    # ── Tax / fiscal ───────────────────────────────────────────────────
    'DBG': 'DBG',    'LIFD': 'DBG',                  # Direkte Bundessteuer
    'MWSTG': 'MWSTG','LTVA': 'MWSTG',                # Mehrwertsteuer / TVA
    'StHG': 'StHG',  'LHID': 'StHG',                 # Steuerharmonisierung
    'VStG': 'VStG',  'LIA': 'VStG',                  # Verrechnungssteuer
    # ── Financial-markets ──────────────────────────────────────────────
    'FusG': 'FusG',  'LFus': 'FusG',                 # Fusionsgesetz
    'KAG': 'KAG',    'LPCC': 'KAG',                  # Kollektivanlagen
    'BankG': 'BankG','LB': 'BankG',                  # Bankgesetz
    'FINIG': 'FINIG','LEFin': 'FINIG',               # Finanzinstitutsgesetz
    'FIDLEG': 'FIDLEG','LSFin': 'FIDLEG',            # Finanzdienstleistungsgesetz
    'FinfraG': 'FinfraG','LIMF': 'FinfraG',          # Finanzmarktinfrastrukturgesetz
    # ── Environment / other ────────────────────────────────────────────
    'USG': 'USG',    'LPE': 'USG',                   # Umweltschutzgesetz
    'ParlG': 'ParlG','LParl': 'ParlG',               # Parlamentsgesetz
}

# ── Procedural-heading patterns (regex over normalised title heading) ──────
# Each pattern flags rows whose role in any code is procedural-apparatus.
# Patterns are written in lower-case; matched case-insensitively after umlaut
# folding (ä→a, ö→o, ü→u).
PROCEDURAL_HEADING_PATTERNS = [
    r'beschwerdefrist',                  # appeal deadline (most universal — appears in 9/10 val golds via BGG 100)
    r'\bbeschwerde\b',                   # appeal mechanism
    r'rechtsmittel',                     # legal remedy (Berufung, Beschwerde, Revision)
    r'berufung',                         # civil appeal
    r'revision',                         # revision
    r'verfahrenskosten',                 # procedural costs
    r'gerichtskosten',                   # court costs
    r'verteidigung',                     # criminal defence
    r'unentgeltliche\s+rechts',          # legal aid
    r'zustandig',                        # jurisdiction (umlaut-folded)
    r'rechtliches\s+gehor',              # right to be heard (umlaut-folded)
    r'grundrechte',                      # fundamental rights (Cst. chapter)
    r'grundsatze\s+des',                 # principles of (criminal) procedure
    r'anfechtbar',                       # appealable decisions
    r'subsidiare\s+verfassungs',         # subsidiary constitutional appeal (BGG 113)
    r'beschwerde\s+in\s+offentlich',     # public-law appeal route (BGG 82)
    r'beschwerde\s+in\s+straf',          # criminal appeal route
    r'beschwerde\s+in\s+zivilsachen',    # civil appeal route (BGG 72)
    r'rechtspflegeverfahren',            # judicial-review procedure (social insurance)
]

# ── Legal-area → code-family ──────────────────────────────────────────────
# Coarse mapping. BGG is included in every area because federal appeals always
# end at the Bundesgericht. The LLM chooses ONE primary area + any secondaries.
LEGAL_AREA_CODES = {
    'criminal-procedure':  ['StPO', 'StGB', 'StBOG', 'BGG', 'BV', 'MStG', 'MStP', 'JStG', 'JStPO'],
    'criminal-substantive':['StGB', 'StPO', 'StBOG', 'BGG', 'BV'],
    'civil-contract':      ['OR', 'ZGB', 'ZPO', 'BGG'],
    'civil-family':        ['ZGB', 'ZPO', 'BGG', 'OR'],
    'civil-inheritance':   ['ZGB', 'OR', 'ZPO', 'BGG'],
    'civil-property':      ['ZGB', 'OR', 'IPRG', 'BGG', 'ZPO'],
    'civil-tort':          ['OR', 'ZGB', 'BGG', 'ZPO'],
    'social-insurance':    ['ATSG', 'IVG', 'AHVG', 'UVG', 'BVG', 'KVG', 'EOG', 'AVIG', 'FZG', 'BGG', 'BV'],
    'public-law':          ['BV', 'BGG', 'VwVG'],
    'constitutional':      ['BV', 'BGG'],
    'debt-enforcement':    ['SchKG', 'OR', 'ZPO', 'BGG', 'ZGB'],
    'tax':                 ['DBG', 'MWSTG', 'StHG', 'VStG', 'BGG', 'BV'],
    'banking':             ['BankG', 'FINIG', 'FIDLEG', 'FinfraG', 'KAG', 'OR', 'ZPO', 'BGG'],
    'environment':         ['USG', 'BGG', 'BV'],
    'international-private':['IPRG', 'ZGB', 'OR', 'BGG'],
}

log(f'CODE_ALIASES: {len(CODE_ALIASES)} entries  ({len(set(CODE_ALIASES.values()))} unique canonical codes)')
log(f'PROCEDURAL_HEADING_PATTERNS: {len(PROCEDURAL_HEADING_PATTERNS)} patterns')
log(f'LEGAL_AREA_CODES: {len(LEGAL_AREA_CODES)} areas')

---

## 3. Utilities

In [ ]:
_TOKEN_RE = re.compile(r'[^\w\d]+', re.UNICODE)

def fold_umlauts(s):
    """ä→a, ö→o, ü→u, ß→ss — for case/diacritic-insensitive heading matching."""
    return (s.lower()
            .replace('ä','a').replace('ö','o').replace('ü','u')
            .replace('ß','ss'))

def tokenise(text):
    if not text: return []
    return [w for w in _TOKEN_RE.split(text.lower().strip()) if len(w) >= 2 or w.isdigit()]

# Parse a citation string into (article_no, abs_no, lit, code).
_CIT_RE = re.compile(
    r'^Art\.\s+(\d+[a-z]?)(?:\s+Abs\.\s+(\d+[a-z]*))?(?:\s+lit\.\s+(\S+))?\s+(\S+)$')

def parse_citation(c):
    m = _CIT_RE.match(c.strip())
    if not m: return None
    return {'art': m.group(1), 'abs': m.group(2), 'lit': m.group(3), 'code': m.group(4)}

def extract_heading(title):
    """laws_de title format: '<Law full name> - <heading>'. Return heading or ''."""
    if not isinstance(title, str): return ''
    if ' - ' not in title: return ''
    h = title.split(' - ', 1)[1].strip()
    # Strip leading numeric/letter prefixes: '3. ', '1. Kapitel: ', 'II. ', 'A. ', 'a. ', 'b. '
    h = re.sub(r'^[0-9]+\.\s+', '', h)
    h = re.sub(r'^[IVXLCDM]+\.\s+', '', h)
    h = re.sub(r'^[A-Za-z]\.\s+', '', h)
    h = re.sub(r'^\d+\s+(Kapitel|Abschnitt|Teil|Titel):\s+', '', h)
    # Strip trailing footnote digits
    h = re.sub(r'\d+$', '', h).strip()
    return h

# Statute-mention regex — handles DE / FR / IT / EN forms
# Examples it captures:
#   Art. 221 Abs. 1 lit. b StPO    (DE)
#   art. 221 al. 1 let. b CPP      (FR)
#   art. 221 cpv. 1 lett. b CPP    (IT)
#   Article 221(1)(b) StPO          (EN)
#   Art. 17 LAI / Art. 100 BGG     (cross-lingual)
_STATUTE_RE = re.compile(
    r'(?:Art(?:icle|\.?)?|art(?:\.|icolo)?)\s*'
    r'(\d+[a-z]?)'
    r'(?:\s*(?:Abs\.?|al\.?|cpv\.?|\()\s*(\d+[a-z]*))?'
    r'(?:\s*\)?\s*(?:lit\.?|let\.?|lett\.?|\()\s*([a-z]+))?'
    r'(?:\s*\)?)?'
    r'\s+(?:de\s+la\s+|du\s+|della?\s+|of\s+the\s+)?'
    r'([A-Z][A-Za-z]{1,8})',
    re.IGNORECASE)

def parse_statute_mentions(text):
    """Return canonical Swiss-German citation strings extracted from query text.

    Uses CODE_ALIASES to canonicalize the code. The lit. suffix is dropped
    because the corpus rarely stores it (laws_de keys at Abs. granularity).
    """
    out = []
    for m in _STATUTE_RE.finditer(text):
        art, ab, lit, code = m.group(1), m.group(2), m.group(3), m.group(4)
        canon_code = CODE_ALIASES.get(code, None)
        if canon_code is None:
            # Try case-insensitive lookup
            for k, v in CODE_ALIASES.items():
                if k.lower() == code.lower():
                    canon_code = v; break
        if canon_code is None:
            continue
        s = f'Art. {art}'
        if ab: s += f' Abs. {ab}'
        s += f' {canon_code}'
        out.append(s)
    # Dedup while preserving order
    seen = set(); deduped = []
    for s in out:
        if s not in seen:
            seen.add(s); deduped.append(s)
    return deduped

# ── Self-test ──────────────────────────────────────────────────────────────
_test = 'Art. 221 Abs. 1 lit. b StPO and art. 221 al. 1 let. b CPP, also Art. 17 LAI, Art. 100 BGG.'
log('Statute parser self-test:', parse_statute_mentions(_test))
log('Heading extraction sample:', extract_heading('Schweizerisches Zivilgesetzbuch vom 10. Dezember 1907 - 3. Eigenhändige Verfügung'))

---

## 4. Load corpus + indices

In [ ]:
section('STAGE 0: LOADING LAWS_DE.CSV + INDEXING')

laws = pd.read_csv(DATA_DIR / 'laws_de.csv')
log(f'  laws_de.csv: {len(laws):,} rows')

# Drop empty-citation rows defensively
laws = laws[laws['citation'].notna() & (laws['citation'].str.len() > 0)].reset_index(drop=True)
log(f'  after dropna: {len(laws):,} rows')

# ── Primary indices ───────────────────────────────────────────────────────
CITATION_TO_ROW   = {c: i for i, c in enumerate(laws['citation'])}
CITATION_LOWER    = {c.lower(): c for c in laws['citation']}     # case-fold lookup
CITATION_UPPER    = {c.upper(): c for c in laws['citation']}     # uppercase-fold lookup

# ── Code → list of citation rows (for area-scoped retrieval) ──────────────
CODE_TO_CITATIONS = defaultdict(list)
for c in laws['citation']:
    parts = c.strip().split()
    if parts:
        CODE_TO_CITATIONS[parts[-1]].append(c)

# ── Heading → list of citation rows ────────────────────────────────────────
HEADING_TO_CITATIONS = defaultdict(list)
HEADING_NORM         = {}    # citation → normalised heading (lower + umlaut-folded)
for c, t in zip(laws['citation'], laws['title']):
    h = extract_heading(t)
    if h:
        HEADING_TO_CITATIONS[h].append(c)
        HEADING_NORM[c] = fold_umlauts(h)

# ── Proper-case map for the casing fix ─────────────────────────────────────
PROPER_CASE_MAP = {c.upper(): c for c in laws['citation']}

# ── Children index for parent-citation fan-out ─────────────────────────────
CHILDREN_INDEX = defaultdict(list)
for c in laws['citation']:
    p = parse_citation(c)
    if p:
        CHILDREN_INDEX[(p['art'], p['code'])].append(c)

log(f'  unique codes (last-token):  {len(CODE_TO_CITATIONS):,}')
log(f'  unique headings:            {len(HEADING_TO_CITATIONS):,}')
log(f'  children index entries:     {len(CHILDREN_INDEX):,}')

# Quick look at the most-frequent headings (these are the doctrine vocabulary)
log('\n  Top 15 most-common headings in laws_de:')
for h, cs in sorted(HEADING_TO_CITATIONS.items(), key=lambda x: -len(x[1]))[:15]:
    log(f'    {len(cs):>5}  {h[:60]}')

In [ ]:
# ── Build / load the title-column BM25 index ──────────────────────────────
TITLE_BM25_PATH = CACHE_DIR / 'title_bm25.pkl'
TITLE_IDS_PATH  = CACHE_DIR / 'title_bm25_ids.pkl'

if TITLE_BM25_PATH.exists() and TITLE_IDS_PATH.exists():
    log('  Loading cached title-BM25 index ...')
    with open(TITLE_BM25_PATH, 'rb') as f: TITLE_BM25 = pickle.load(f)
    with open(TITLE_IDS_PATH,  'rb') as f: TITLE_BM25_IDS = pickle.load(f)
else:
    log('  Building title-BM25 index from scratch ...')
    title_tokens = []
    for t in tqdm(laws['title'], desc='  Tokenising titles'):
        title_tokens.append(tokenise(t if isinstance(t, str) else ''))
    TITLE_BM25     = BM25Okapi(title_tokens)
    TITLE_BM25_IDS = list(laws['citation'])
    with open(TITLE_BM25_PATH, 'wb') as f: pickle.dump(TITLE_BM25, f)
    with open(TITLE_IDS_PATH,  'wb') as f: pickle.dump(TITLE_BM25_IDS, f)
    log('  ✓ Saved.')

# ── Build / load the text-column BM25 index (safety-net base channel) ─────
BODY_BM25_PATH = CACHE_DIR / 'body_bm25.pkl'
BODY_IDS_PATH  = CACHE_DIR / 'body_bm25_ids.pkl'

if BODY_BM25_PATH.exists() and BODY_IDS_PATH.exists():
    log('  Loading cached body-BM25 index ...')
    with open(BODY_BM25_PATH, 'rb') as f: BODY_BM25 = pickle.load(f)
    with open(BODY_IDS_PATH,  'rb') as f: BODY_BM25_IDS = pickle.load(f)
else:
    log('  Building body-BM25 index from scratch ...')
    body_tokens = []
    for t in tqdm(laws['text'], desc='  Tokenising bodies'):
        body_tokens.append(tokenise(t if isinstance(t, str) else ''))
    BODY_BM25     = BM25Okapi(body_tokens)
    BODY_BM25_IDS = list(laws['citation'])
    with open(BODY_BM25_PATH, 'wb') as f: pickle.dump(BODY_BM25, f)
    with open(BODY_IDS_PATH,  'wb') as f: pickle.dump(BODY_BM25_IDS, f)
    log('  ✓ Saved.')

log(f'  title-BM25 docs: {len(TITLE_BM25_IDS):,}')
log(f'  body-BM25 docs:  {len(BODY_BM25_IDS):,}')

In [ ]:
# ── Load val (for diagnostic eval). Test queries are loaded similarly. ─────
val = pd.read_csv(DATA_DIR / 'val.csv')
log(f'val.csv: {len(val):,} queries')

def extract_law_gold(gold_str):
    """Pull the Art.-prefixed citation strings out of a gold_citations cell."""
    raw = [c.strip() for c in str(gold_str or '').split(';') if c.strip()]
    return [c for c in raw if re.match(r'^Art\.\s+\d', c)]

if 'gold_citations' in val.columns:
    val_law_gold = {row['query_id']: extract_law_gold(row['gold_citations'])
                    for _, row in val.iterrows()}
    n_law = sum(len(v) for v in val_law_gold.values())
    n_in  = sum(1 for v in val_law_gold.values() for c in v if c in CITATION_TO_ROW)
    log(f'  val law gold: {n_law} total, {n_in} present in laws_de.csv ({100*n_in/max(n_law,1):.1f}%)')
else:
    val_law_gold = {}

---

## 5. LLM query expander (Qwen3-8B)

One Qwen3-8B call per query produces a structured expansion used by Channels 2 and 3:

```json
{
  "legal_area":            "<one of LEGAL_AREA_CODES keys>",
  "secondary_areas":       ["..."],
  "statute_mentions":      ["Art. 221 Abs. 1 lit. b StPO", ...],
  "doctrine_concepts_de":  ["Untersuchungshaft", "Beschwerdefrist", "Pflichtteil", ...],
  "german_query":          "<paraphrase of the query in formal Swiss legal German>"
}
```

Cached to Drive as `llm_expansions.json`. The cache is keyed by query_id; running on test queries reuses any val expansions you already have.


In [ ]:
EXPANDER_SYSTEM = '''You are a Swiss Federal Court (Bundesgericht) legal citation expert with deep knowledge of Swiss federal law in German, French, and Italian.

Given an English query describing a Swiss-law scenario, output a JSON object with these exact fields:

{
  "legal_area": one of ["criminal-procedure", "criminal-substantive", "civil-contract", "civil-family", "civil-inheritance", "civil-property", "civil-tort", "social-insurance", "public-law", "constitutional", "debt-enforcement", "tax", "banking", "environment", "international-private"],
  "secondary_areas": [list of additional areas from the same vocabulary that the query also touches; empty list if none],
  "statute_mentions": [list of statute references appearing verbatim in the query, normalised to the form "Art. N Abs. M CODE" using the canonical Swiss-German abbreviation (StPO, StGB, ZGB, OR, IVG, ATSG, BGG, BV, StBOG, ZPO, SchKG, IPRG, ...)],
  "doctrine_concepts_de": [up to 30 German doctrine names that govern this legal scenario, drawn from canonical Swiss legal vocabulary — e.g. "Untersuchungshaft", "Kollusionsgefahr", "Verhältnismässigkeit", "Beschwerdefrist", "Verfahrenskosten", "Pflichtteil", "Eigenhändige Verfügung", "Testierfähigkeit", "Urteilsfähigkeit", "Mangelhafter Wille", "Guter Glaube", "Erwerb in gutem Glauben", "Besitzanweisung", "Persönlicher Verkehr", "Besuchsrecht", "Erwerbsunfähigkeit", "Invalidität", "Hypothetisches Einkommen", "Erwerbsersatz", "Berufliche Massnahmen", "Verschulden", "Adaequater Kausalzusammenhang", "Haftung für Hilfspersonen", "Wegbedingung der Haftung", "Auslegung der Verträge", "Vertrauensprinzip", "Gefälligkeit", "Werkvertrag", "Auftrag", "Schenkung", "Betreibungsbegehren", "Anweisung an die Schuldner", "Rechtliches Gehör", "Treu und Glauben"],
  "german_query": "a 2-3 sentence paraphrase of the query in formal Swiss legal German, mentioning the relevant statutes, codes, doctrines, and procedural concepts"
}

Rules:
- Output ONLY the JSON object, nothing else. No explanation, no preamble, no markdown fences.
- Use Swiss-German conventions (StPO not CPP, ZGB not CC, BGG not LTF, even if the query uses the French/Italian variant).
- If the query mentions a court appeal (any), the area is the substantive area of the dispute, not "public-law". Mark BGG-related procedural concepts in doctrine_concepts_de.
- doctrine_concepts_de should be specific (Untersuchungshaft, not just Haft).
- Be comprehensive in doctrine_concepts_de — list every concept the gold-annotating expert would think of.
'''

def load_expander(model_name=EXPANDER_MODEL, device=DEVICE):
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    log(f'  Loading expander: {model_name}')
    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.bfloat16, device_map=device, trust_remote_code=True)
    model.eval()
    log(f'  ✓ VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')
    return tok, model

def _strip_think(raw):
    if '</think>' in raw:
        raw = raw.split('</think>')[-1]
    return raw.strip()

def _extract_json(raw):
    """Best-effort: pull the first {...} object out of model output."""
    s = _strip_think(raw)
    # Strip code fences if present
    s = re.sub(r'^```(?:json)?\s*', '', s).rstrip('`').strip()
    # Find balanced braces
    depth = 0; start = None
    for i, ch in enumerate(s):
        if ch == '{':
            if depth == 0: start = i
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0 and start is not None:
                blob = s[start:i+1]
                try:
                    return json.loads(blob)
                except json.JSONDecodeError:
                    pass
    return None

def expand_query(query_en, tok, model):
    import torch
    msgs = [{'role':'system','content':EXPANDER_SYSTEM},
            {'role':'user','content':f'QUERY:\n{query_en}'}]
    text = tok.apply_chat_template(msgs, tokenize=False,
                                   add_generation_prompt=True, enable_thinking=True)
    inputs = tok(text, return_tensors='pt', truncation=True, max_length=12000).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=2000,
                             do_sample=True, temperature=0.3, top_k=20, top_p=0.95,
                             pad_token_id=tok.eos_token_id)
    gen = out[0][inputs['input_ids'].shape[1]:]
    raw = tok.decode(gen, skip_special_tokens=True)
    parsed = _extract_json(raw)
    if parsed is None:
        log(f'  WARNING: failed to parse JSON from expander; using fallback empty record')
        parsed = {'legal_area':'civil-contract','secondary_areas':[],
                  'statute_mentions':[],'doctrine_concepts_de':[],
                  'german_query': query_en}
    return parsed, raw[:300]

def run_expander(queries, cache_path):
    """queries: list of (qid, query_en). Returns dict {qid: expansion}."""
    cache = {}
    if Path(cache_path).exists():
        with open(cache_path, encoding='utf-8') as f:
            cache = json.load(f)
        log(f'  Loaded expansion cache: {len(cache)} entries')
    needed = [(q, t) for q, t in queries if q not in cache]
    if not needed:
        log('  ✓ All expansions cached.')
        return cache
    tok, model = load_expander()
    for qid, qtxt in tqdm(needed, desc='  Expanding'):
        exp, raw = expand_query(qtxt, tok, model)
        cache[qid] = exp
    import torch
    del model, tok; gc.collect(); torch.cuda.empty_cache()
    Path(cache_path).parent.mkdir(parents=True, exist_ok=True)
    with open(cache_path, 'w', encoding='utf-8') as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)
    log(f'  ✓ Saved expansion cache ({len(cache)})')
    return cache

---

## 6. Channel 1 — Statute parser

Extracts `Art. N (Abs. M)? CODE` from the query and LLM's `statute_mentions`. Canonicalises code via `CODE_ALIASES`. Direct lookup in `laws_de.csv`. Parent citations (`Art. 221 StPO` with no `Abs.`) fan out to children when the corpus stores at Abs-level.

In [ ]:
def channel_1_statute(query_en, expansion):
    """Return list of citation strings (corpus-canonical) from statute parsing."""
    mentions = set(parse_statute_mentions(query_en))
    for s in expansion.get('statute_mentions', []) or []:
        mentions.update(parse_statute_mentions(s))
        # If the LLM-formatted mention is already canonical, accept it directly:
        if s in CITATION_TO_ROW:
            mentions.add(s)

    hits = set()
    for m in mentions:
        # Direct lookup
        if m in CITATION_TO_ROW:
            hits.add(m); continue
        # Case-insensitive
        if m.lower() in CITATION_LOWER:
            hits.add(CITATION_LOWER[m.lower()]); continue
        if m.upper() in CITATION_UPPER:
            hits.add(CITATION_UPPER[m.upper()]); continue
        # Parent (Art. N CODE with no Abs.) → fan out to all Abs. children
        p = parse_citation(m)
        if p and not p['abs']:
            children = CHILDREN_INDEX.get((p['art'], p['code']), [])
            for c in children:
                hits.add(c)
    return sorted(hits)

# Quick self-test on val_001
if val_law_gold:
    q1 = val.iloc[0]['query']
    log('Channel 1 on val_001 (without LLM yet — only regex over query):')
    h = channel_1_statute(q1, {'statute_mentions': []})
    for c in h: log(f'  {c}')

---

## 7. Channel 2 — Title-column BM25 + heading dictionary

Two paths:

- **2a (title-BM25):** BM25 over the `title` column using `expansion.german_query` as the query (and the English query as backup). Top-K.
- **2b (doctrine-heading dictionary):** for each German doctrine concept the LLM produced, retrieve every `laws_de` row whose normalised heading contains that token. Limits per-concept to avoid pool explosion.

In [ ]:
def channel_2_title_bm25(expansion, top_k=TITLE_BM25_TOP_K):
    """BM25 over the title column. Returns list of (citation, rank)."""
    gq = expansion.get('german_query', '') or ''
    concepts = ' '.join(expansion.get('doctrine_concepts_de', []) or [])
    tokens = tokenise(gq + ' ' + concepts)
    if not tokens:
        return []
    scores = TITLE_BM25.get_scores(tokens)
    order  = scores.argsort()[::-1][:top_k]
    return [(TITLE_BM25_IDS[i], int(rank+1), float(scores[i])) for rank, i in enumerate(order)]

def channel_2_doctrine_dict(expansion, per_concept_cap=50):
    """For each doctrine concept, find laws_de rows whose heading matches it.

    Match is substring (umlaut-folded, lowercased) — the heading vocabulary is
    closed (mined from laws_de.csv) so this is cheap and precise.
    """
    out = defaultdict(list)
    for concept in expansion.get('doctrine_concepts_de', []) or []:
        needle = fold_umlauts(concept.strip())
        if len(needle) < 4:
            continue
        matches = []
        for c, norm in HEADING_NORM.items():
            if needle in norm:
                matches.append(c)
                if len(matches) >= per_concept_cap:
                    break
        for m in matches:
            out[m].append(concept)
    return out  # citation -> list of concepts that matched

def channel_2(expansion):
    bm25_hits   = channel_2_title_bm25(expansion)         # ordered
    bm25_set    = {c for c, _, _ in bm25_hits}
    dict_hits   = channel_2_doctrine_dict(expansion)      # dict citation->concepts
    return {'title_bm25': bm25_hits, 'doctrine_dict': dict_hits, 'bm25_set': bm25_set}

---

## 8. Channel 3 — Procedural apparatus

Given the LLM's `legal_area`, scope to the relevant codes via `LEGAL_AREA_CODES`. Then keep only rows whose `title` heading matches one of the universal `PROCEDURAL_HEADING_PATTERNS`. These articles are gold for any appeal in that area by structural necessity.

In [ ]:
_PROC_RE = re.compile('|'.join(PROCEDURAL_HEADING_PATTERNS), re.IGNORECASE)

def channel_3_procedural(expansion):
    """Return list of citations matching (legal-area code family) AND (procedural heading)."""
    areas = [expansion.get('legal_area')] + (expansion.get('secondary_areas', []) or [])
    codes = set()
    for a in areas:
        if a in LEGAL_AREA_CODES:
            codes.update(LEGAL_AREA_CODES[a])
    if not codes:
        codes = {'BGG'}   # always include BGG as the appeal apparatus

    hits = set()
    for code in codes:
        for c in CODE_TO_CITATIONS.get(code, []):
            h = HEADING_NORM.get(c, '')
            if h and _PROC_RE.search(h):
                hits.add(c)
    return sorted(hits)

# Self-test (no LLM yet — fake an expansion)
_fake = {'legal_area':'criminal-procedure', 'secondary_areas':[], 'doctrine_concepts_de':[], 'german_query':''}
log(f'Channel 3 on criminal-procedure (fake expansion): {len(channel_3_procedural(_fake))} hits')

---

## 9. Channel 4 — One-hop reference expansion

For each citation that Channels 1, 2, or 3 surface, regex the candidate's `text` body for `Art. N (Abs. M)? CODE` references, look those up in `laws_de.csv`, and add as Tier-2 candidates. Depth is capped at 1 hop.

In [ ]:
# Build laws_de text lookup
LAW_TEXT = {c: (t if isinstance(t, str) else '')
            for c, t in zip(laws['citation'], laws['text'])}

def channel_4_refs(seed_citations):
    """Regex Art. N CODE in each seed's body; resolve via CODE_ALIASES; lookup."""
    out = set()
    for sc in seed_citations:
        body = LAW_TEXT.get(sc, '')
        if not body: continue
        for m in parse_statute_mentions(body):
            if m in CITATION_TO_ROW:
                out.add(m); continue
            # Parent fanout
            p = parse_citation(m)
            if p and not p['abs']:
                for ch in CHILDREN_INDEX.get((p['art'], p['code']), []):
                    out.add(ch)
    return sorted(out - set(seed_citations))

---

## 10. Body BM25 — safety-net base channel

Pure BM25 over the `text` column with English query + German expansion. Caps the recall floor in case the LLM expansion misses a doctrine name.

In [ ]:
def channel_body_bm25(query_en, expansion, top_k=BODY_BM25_TOP_K):
    gq = expansion.get('german_query', '') or ''
    concepts = ' '.join(expansion.get('doctrine_concepts_de', []) or [])
    tokens = tokenise(query_en + ' ' + gq + ' ' + concepts)
    if not tokens: return []
    scores = BODY_BM25.get_scores(tokens)
    order  = scores.argsort()[::-1][:top_k]
    return [(BODY_BM25_IDS[i], int(rank+1), float(scores[i])) for rank, i in enumerate(order)]

---

## 11. Candidate pool builder

Unions all channel outputs into a single `{citation: {channels, ranks, ...}}` map per query.

In [ ]:
@dataclass
class Candidate:
    citation:         str
    channels:         set = field(default_factory=set)    # which channels surfaced it
    title_bm25_rank:  int = 10**9
    title_bm25_score: float = 0.0
    body_bm25_rank:   int = 10**9
    body_bm25_score:  float = 0.0
    matched_doctrine: list = field(default_factory=list)
    reranker_score:   float = 0.0

def build_pool(query_en, expansion):
    pool = {}
    def get(cit):
        if cit not in pool: pool[cit] = Candidate(citation=cit)
        return pool[cit]

    # Channel 1 — statute parser
    for c in channel_1_statute(query_en, expansion):
        get(c).channels.add('statute_parser')

    # Channel 2 — title BM25 + doctrine dict
    ch2 = channel_2(expansion)
    for c, rank, score in ch2['title_bm25']:
        cd = get(c); cd.channels.add('title_bm25')
        cd.title_bm25_rank = min(cd.title_bm25_rank, rank); cd.title_bm25_score = max(cd.title_bm25_score, score)
    for c, concepts in ch2['doctrine_dict'].items():
        cd = get(c); cd.channels.add('doctrine_dict'); cd.matched_doctrine.extend(concepts)

    # Channel 3 — procedural apparatus
    for c in channel_3_procedural(expansion):
        get(c).channels.add('procedural_apparatus')

    # Body BM25 — base safety net (gets its own channel name)
    for c, rank, score in channel_body_bm25(query_en, expansion):
        cd = get(c); cd.channels.add('body_bm25')
        cd.body_bm25_rank = min(cd.body_bm25_rank, rank); cd.body_bm25_score = max(cd.body_bm25_score, score)

    # Channel 4 — one-hop refs from whatever the above produced
    seeds = list(pool.keys())
    for c in channel_4_refs(seeds):
        get(c).channels.add('ref_expansion')

    # Hard cap (favour candidates with more channel hits + better BM25 ranks)
    if len(pool) > POOL_HARD_CAP:
        def keyfn(cand):
            return (-len(cand.channels), cand.title_bm25_rank, cand.body_bm25_rank)
        kept = sorted(pool.values(), key=keyfn)[:POOL_HARD_CAP]
        pool = {c.citation: c for c in kept}

    return pool

---

## 12. Qwen3-Reranker-8B (precision layer)

Scores each (query, candidate) pair as `P(yes) ∈ [0,1]`. Mutates `Candidate.reranker_score`. Cached on Drive.

In [ ]:
def _model_slug(name): return re.sub(r'[^a-zA-Z0-9_-]', '_', name)

RERANKER_INSTRUCTION = ('Given a query about Swiss law, determine whether the provided '
                       'law article is related to or applicable to the legal issue.')

def load_reranker(model_name=RERANKER_MODEL, device=DEVICE):
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    log(f'  Loading reranker: {model_name}')
    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, padding_side='left')
    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.bfloat16, device_map=device, trust_remote_code=True)
    model.eval()
    prefix = tok.encode(
        '<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n',
        add_special_tokens=False)
    suffix = tok.encode(
        '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n',
        add_special_tokens=False)
    yes_id = tok.convert_tokens_to_ids('yes')
    no_id  = tok.convert_tokens_to_ids('no')
    return tok, model, prefix, suffix, yes_id, no_id

def _format_pair(query_en, cit):
    text = LAW_TEXT.get(cit, '')
    # title row also retrievable from the laws df (re-index for speed)
    row = laws.iloc[CITATION_TO_ROW[cit]] if cit in CITATION_TO_ROW else None
    title = row['title'] if row is not None else ''
    doc = f'{cit}\nTitle: {title}\nText: {text[:500]}'
    return f'<Instruct>: {RERANKER_INSTRUCTION}\n<Query>: {query_en}\n<Document>: {doc}'

def rerank_pool(query_en, pool, tok, model, prefix, suffix, yes_id, no_id,
                batch_size=RERANKER_BATCH, max_length=4096):
    import torch
    cits = list(pool.keys())
    pairs = [_format_pair(query_en, c) for c in cits]
    scores = []
    for i in range(0, len(pairs), batch_size):
        batch = pairs[i:i+batch_size]
        inputs = tok(batch, padding=False, truncation='longest_first',
                     return_attention_mask=False,
                     max_length=max_length - len(prefix) - len(suffix))
        for j in range(len(inputs['input_ids'])):
            inputs['input_ids'][j] = prefix + inputs['input_ids'][j] + suffix
        inputs = tok.pad(inputs, padding=True, return_tensors='pt')
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        with torch.no_grad():
            logits = model(**inputs).logits[:, -1, :]
        stacked = torch.stack([logits[:, no_id], logits[:, yes_id]], dim=1)
        p_yes = torch.nn.functional.log_softmax(stacked, dim=1)[:, 1].exp().cpu().tolist()
        scores.extend(p_yes)
    for c, s in zip(cits, scores):
        pool[c].reranker_score = float(s)

def run_reranker(all_pools, query_texts, cache_path):
    """Score every (qid, candidate) and write to cache. all_pools = {qid: pool}"""
    cache = {}
    if Path(cache_path).exists():
        try:
            with open(cache_path, encoding='utf-8') as f: cache = json.load(f)
        except Exception: pass
    log(f'  Reranker cache: {len(cache)} qids loaded')
    needed = [qid for qid in all_pools if qid not in cache or set(cache[qid]) != set(all_pools[qid])]
    if needed:
        tok, model, prefix, suffix, yes_id, no_id = load_reranker()
        for qid in tqdm(needed, desc='  Reranking'):
            rerank_pool(query_texts[qid], all_pools[qid], tok, model, prefix, suffix, yes_id, no_id)
            cache[qid] = {c: pool_cand.reranker_score for c, pool_cand in all_pools[qid].items()}
        import torch
        del model, tok; gc.collect(); torch.cuda.empty_cache()
        with open(cache_path, 'w', encoding='utf-8') as f:
            json.dump(cache, f, ensure_ascii=False, indent=2)
        log(f'  ✓ Saved reranker cache ({len(cache)})')
    else:
        log('  ✓ All cached.')
    # Backfill from cache to live pools
    for qid, pool in all_pools.items():
        for c in pool:
            pool[c].reranker_score = float(cache.get(qid, {}).get(c, 0.0))
    return cache

---

## 13. Multi-channel selection

Per-channel reliability dictates the threshold:

- **statute_parser** — exact-match against query: always include.
- **procedural_apparatus** — structural appeal articles for the area: include if reranker > 0.30.
- **title_bm25** — heading-driven lexical match: include if reranker > 0.50.
- **doctrine_dict** — heading substring match against LLM doctrine list: include if reranker > 0.50.
- **ref_expansion** — one-hop body reference: include if reranker > 0.55.
- **body_bm25 only** — soft signal: include only if reranker > 0.70.
- **2+ channels** — multi-signal: include if reranker > 0.40.

In [ ]:
def select_predictions(pool):
    selected = set()
    for c, cand in pool.items():
        s = cand.reranker_score
        chs = cand.channels
        n   = len(chs)

        # 1) Always-include channels (exact match)
        if any(ch in TH_ALWAYS_INCLUDE for ch in chs):
            selected.add(c); continue
        # 2) Procedural apparatus + reranker confirms
        if 'procedural_apparatus' in chs and s >= TH_PROCEDURAL:
            selected.add(c); continue
        # 3) Multi-channel
        if n >= MULTI_CHANNEL_MIN_HITS and s >= TH_MULTI_CHANNEL:
            selected.add(c); continue
        # 4) Single-channel — channel-specific thresholds
        if 'title_bm25' in chs and s >= TH_TITLE_BM25:
            selected.add(c); continue
        if 'doctrine_dict' in chs and s >= TH_TITLE_BM25:
            selected.add(c); continue
        if 'ref_expansion' in chs and s >= TH_REF_EXPANSION:
            selected.add(c); continue
        # 5) Reranker only, very high confidence
        if s >= TH_SOLO_RERANKER:
            selected.add(c); continue
    return selected

def apply_proper_case(preds):
    return {qid: [PROPER_CASE_MAP.get(c.upper(), c) for c in cs] for qid, cs in preds.items()}

---

## 14. Official scorer (mirrors `scripts/evaluate_submission.py`)

In [ ]:
_WS_RE = re.compile(r'\s+')
def _canon(c): return _WS_RE.sub(' ', c.strip())

def _parse_field(v):
    if pd.isna(v): return set()
    if not isinstance(v, str): v = str(v)
    parts = v.split(';')
    return {_canon(p) for p in parts if _canon(p)}

def _prf1(p, g):
    if not p and not g: return 1.0, 1.0, 1.0
    if not p or not g:  return 0.0, 0.0, 0.0
    tp = len(p & g)
    P, R = tp/len(p), tp/len(g)
    F = (2*P*R/(P+R)) if (P+R)>0 else 0.0
    return P, R, F

def evaluate(preds, df, label='', per_query=True, law_only=True):
    """If law_only=True, restrict gold to Art.-prefixed citations (drops court)."""
    if per_query:
        log(f'\n  ── {label} ──')
        log(f'  {"QID":<14} {"P":>6} {"R":>6} {"F1":>6} {"Pred":>5} {"Gold":>5}')
        log('-'*60)
    ps, rs, fs = [], [], []
    for _, row in df.iterrows():
        qid = row['query_id']
        gold = _parse_field(row.get('gold_citations', ''))
        if law_only:
            gold = {c for c in gold if re.match(r'^Art\.\s+\d', c)}
        if not gold: continue
        pred = _parse_field(';'.join(preds.get(qid, [])))
        if law_only:
            pred = {c for c in pred if re.match(r'^Art\.\s+\d', c)}
        P, R, F = _prf1(pred, gold)
        ps.append(P); rs.append(R); fs.append(F)
        if per_query:
            log(f'  {qid:<14} {P:6.3f} {R:6.3f} {F:6.3f} {len(pred):5d} {len(gold):5d}')
    if per_query: log('-'*60)
    avg = lambda xs: sum(xs)/max(len(xs),1)
    P, R, F = avg(ps), avg(rs), avg(fs)
    if per_query: log(f'  {"MACRO":>14} {P:6.3f} {R:6.3f} {F:6.3f}')
    return {'p': P, 'r': R, 'f1': F}

---

## 15. End-to-end runner

In [ ]:
def main(target_df=None, label='val'):
    """Run end-to-end on a query DataFrame (val.csv or test.csv)."""
    t0 = time.time()
    if target_df is None:
        target_df = val
    queries = [(row['query_id'], row['query']) for _, row in target_df.iterrows()]
    query_texts = {q: t for q, t in queries}

    # Stage 1: LLM expander
    section(f'STAGE 1: LLM QUERY EXPANSION ({label})')
    expansions = run_expander(queries, CACHE_DIR / 'llm_expansions.json')

    # Stage 2: Build candidate pool per query (all channels)
    section('STAGE 2: BUILD CANDIDATE POOLS')
    all_pools = {}
    for qid, qtxt in tqdm(queries, desc='  Pool'):
        pool = build_pool(qtxt, expansions.get(qid, {}))
        all_pools[qid] = pool
    avg_pool = sum(len(p) for p in all_pools.values()) / max(len(all_pools),1)
    log(f'  avg pool size: {avg_pool:.1f}')

    # Diagnostic: candidate-pool law-recall vs val gold
    if val_law_gold and label == 'val':
        log('\n  Candidate-pool law-recall vs val gold:')
        for qid, gold in val_law_gold.items():
            if qid not in all_pools: continue
            gset = set(gold)
            pset = set(all_pools[qid].keys())
            R = len(gset & pset) / max(len(gset),1)
            missing = sorted(gset - pset)
            tag = '✓' if R == 1.0 else f'MISS {len(missing)}'
            log(f'    {qid}: R={R:.3f} pool={len(pset)} gold={len(gset)} {tag}')
            for m in missing[:8]:
                log(f'        MISSING: {m}')

    # Stage 3: Reranker
    section('STAGE 3: RERANKER')
    run_reranker(all_pools, query_texts, CACHE_DIR / 'reranker_cache.json')

    # Stage 4: Select + casing fix
    section('STAGE 4: SELECTION + CASING')
    preds = {qid: sorted(select_predictions(pool)) for qid, pool in all_pools.items()}
    preds = apply_proper_case(preds)
    log(f'  avg predictions/query: {sum(len(v) for v in preds.values()) / max(len(preds),1):.1f}')

    # Stage 5: Evaluate (only if gold available)
    if 'gold_citations' in target_df.columns and target_df['gold_citations'].notna().any():
        section('STAGE 5: VAL EVALUATION (law-only)')
        evaluate(preds, target_df, label=f'{label} law-only', law_only=True)
        section('STAGE 5: VAL EVALUATION (whole-gold — informational only)')
        evaluate(preds, target_df, label=f'{label} whole-gold', law_only=False)

    # Stage 6: Submission
    sub = pd.DataFrame({
        'query_id': [q for q, _ in queries],
        'predicted_citations': [';'.join(preds.get(q, [])) for q, _ in queries],
    })
    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    out_path = OUTPUT_DIR / f'submission_{label}_{ts}.csv'
    sub.to_csv(out_path, index=False)
    sub.to_csv(OUTPUT_DIR / f'submission_{label}.csv', index=False)
    log(f'\n  ✓ Submission: {out_path}')
    log(f'  ✓ Stable:     {OUTPUT_DIR / f"submission_{label}.csv"}')
    log(f'  Total: {time.time()-t0:.0f}s')
    return preds, all_pools, expansions

In [ ]:
# Run on val (diagnostic — has gold)
preds_val, pools_val, expansions_val = main(target_df=val, label='val')

In [ ]:
# Optional: run on test.csv (no gold; produces submission CSV).
# Uncomment when ready.
# test = pd.read_csv(DATA_DIR / 'test.csv')
# preds_test, pools_test, _ = main(target_df=test, label='test')

---

## 16. Tuning notes

**If law-only recall < 1.0 on val:** look at the per-query `MISSING` lines printed by stage 2. Possible causes:

- **Missing channel** — the gold article isn't surfaced by any channel. Check whether (a) the article's `title` heading is in the doctrine vocab the LLM produced (it should be — if not, the LLM expansion is under-broad; widen the `EXPANDER_SYSTEM` examples), or (b) the article is procedural but its heading didn't match `PROCEDURAL_HEADING_PATTERNS` (add the heading pattern).
- **LLM legal-area mis-classification** — Channel 3 ran on the wrong code family. Look at `expansions_val[qid]['legal_area']`. If wrong, widen the area set or add the area to `secondary_areas` in the prompt examples.

**If law-only precision < target on val:** look at false positives in the per-query eval lines.

- Raise the relevant channel threshold (`TH_TITLE_BM25`, `TH_REF_EXPANSION`, `TH_SOLO_RERANKER`).
- The `procedural_apparatus` channel is the one most likely to introduce false positives. If a non-criminal query is getting StPO procedural articles, the LLM `legal_area` is over-broad — tighten the prompt.

**No val-specific hardcoding.** Every constant table (`CODE_ALIASES`, `PROCEDURAL_HEADING_PATTERNS`, `LEGAL_AREA_CODES`) is sourced from public Swiss Federal Code metadata, not from val gold. Thresholds (`TH_*`) are global hyperparameters tuned to channel reliability, not per-query.
